In [23]:
#!pip install spacy
!pip install spacy datasets

In [24]:
import spacy
from spacy.tokens import DocBin, Span
from datasets import load_dataset

# 1. Carregar os dados
raw_datasets = load_dataset("lfcc/portuguese_ner")
nlp = spacy.blank("pt") 

def convert(dataset, outfile):
    db = DocBin()
    # Obter a lista de nomes das etiquetas (ex: ['O', 'B-PER', 'I-PER', ...])
    label_list = dataset.features["ner_tags"].feature.names
    
    for item in dataset:
        tokens = item["tokens"]
        tags = item["ner_tags"]
        
        # Criar o doc apenas com o texto/tokens
        doc = nlp.make_doc(" ".join(tokens))
        
        entities = []
        current_start = None
        current_label = None
        
        for i, tag_id in enumerate(tags):
            full_label = label_list[tag_id]
            
            # Lógica para agrupar B- e I- em entidades únicas
            if full_label.startswith("B-"):
                # Se já havia uma entidade aberta, fecha-a
                if current_start is not None:
                    entities.append(Span(doc, current_start, i, label=current_label))
                current_start = i
                current_label = full_label[2:] # Remove o "B-"
            
            elif full_label.startswith("I-"):
                # Continua a entidade se o label for o mesmo
                if current_start is None: # Caso o dataset tenha erro e comece com I-
                    current_start = i
                    current_label = full_label[2:]
            
            else: # Tag "O" (Outside)
                if current_start is not None:
                    entities.append(Span(doc, current_start, i, label=current_label))
                    current_start = None
                    current_label = None
        
        # Se terminou o loop e ainda houver uma entidade aberta
        if current_start is not None:
            entities.append(Span(doc, current_start, len(tags), label=current_label))

        try:
            doc.ents = entities
            db.add(doc)
        except Exception as e:
            # Útil para debugar se houver sobreposição de spans
            continue 
            
    db.to_disk(outfile)
    print(f"Ficheiro guardado: {outfile}")

# 3. Gerar os ficheiros
convert(raw_datasets["train"], "train.spacy")
convert(raw_datasets["test"], "dev.spacy")

Ficheiro guardado: train.spacy
Ficheiro guardado: dev.spacy
